In [ ]:
import sys
import community as louvain_community
import networkx as nx
import itertools
import infomap
from collections import Counter

sys.path.append("../../legal-data-clustering/")
%run '../../legal-data-clustering/legal_data_clustering/pipeline/cd_cluster.py'
%run '../../legal-data-clustering/legal_data_clustering/utils/graph_api.py'
%run 'common.py'

import altair.vega.v5 as alt
import cdlib
import numpy as np

# Functions

In [ ]:
def vega_circle_dendrogram_schema(data, default_size_filter, layout="tidy", greyscale=False, rotation_angle=0, default_radius=290):
    return {
      "$schema": "https://vega.github.io/schema/vega/v5.json",
      "width": 720,
      "height": 720,
      "padding": 5,
      "autosize": "none",
      "signals": [
        {
          "name": "radius", "value": default_radius,
          "bind": {"input": "range", "min": 200, "max": 320}
        },
        {
          "name": "min_size", "value": default_size_filter,
          "bind": {"input": "range", "min": default_size_filter, "max": default_size_filter*10}
        },
        {
          "name": "layout", "value": layout,
          "bind": {"input": "radio", "options": ["tidy", "cluster"]}
        },
        {
          "name": "links", "value": "line",
          "bind": {
            "input": "select",
            "options": ["line", "curve", "diagonal", "orthogonal"]
          }
        },
        { "name": "originX", "update": "width / 2" },
        { "name": "originY", "update": "height / 2" },
        { "name": "separation", "value": False, "bind": {"input": "checkbox"} }
      ],

      "data": [
        {
          "name": "tree",
          "values": data,
          "transform": [
            {
              "type": "filter",
              "expr": "datum.size > min_size",
            },
            {
              "type": "stratify",
              "key": "id",
              "parentKey": "parent"
            },
            {
              "type": "tree",
              "method": {"signal": "layout"},
              "size": [1, {"signal": "radius"}],
              "separation": {"signal": "separation"},
              "as": ["alpha", "radius", "depth", "children"]
            },
            {
              "type": "formula",
              "expr": f"(360 * datum.alpha + 270) % 360 + {rotation_angle}",
              "as":   "angle"
            },
            {
              "type": "formula",
              "expr": "PI * datum.angle / 180",
              "as":   "radians"
            },
            {
              "type": "formula",
              "expr": "inrange(datum.angle, [90, 270])",
              "as":   "leftside"
            },
            {
              "type": "formula",
              "expr": "originX + datum.radius * cos(datum.radians)",
              "as":   "x"
            },
            {
              "type": "formula",
              "expr": "originY + datum.radius * sin(datum.radians)",
              "as":   "y"
            }
          ]
        },
        {
          "name": "links",
          "source": "tree",
          "transform": [
            { "type": "treelinks" },
            {
              "type": "linkpath",
              "shape": {"signal": "links"}, "orient": "radial",
              "sourceX": "source.radians", "sourceY": "source.radius",
              "targetX": "target.radians", "targetY": "target.radius"
            }
          ]
        }
      ],

      "scales": [
        {
          "name": "color",
          "type": "linear",
          "range": {"scheme": "greys" if greyscale else "magma"},
          "domain": {"data": "tree", "field": "depth"},
          "zero": True
        }
      ],

      "marks": [
        {
          "type": "path",
          "from": {"data": "links"},
          "encode": {
            "update": {
              "x": {"signal": "originX"},
              "y": {"signal": "originY"},
              "path": {"field": "path"},
              "stroke": {"value": "#000"}
            }
          }
        },
        {
          "type": "symbol",
          "from": {"data": "tree"},
          "encode": {
            "enter": {
              "size": {"value": 100},
              "stroke": {"value": "#fff"}
            },
            "update": {
              "x": {"field": "x"},
              "y": {"field": "y"},
              "fill": {"scale": "color", "field": "depth"}
            }
          }
        },
        {
          "type": "text",
          "from": {"data": "tree"},
          "encode": {
            "enter": {
              "text": {"field": "name"},
              "fontSize": {"value": 10.8},
              "baseline": {"value": "middle"},
              "font": {"value": "Times New Roman"},
              "fontWeight": {"value": "bold"}
            },
            "update": {
              "x": {"field": "x"},
              "y": {"field": "y"},
              "dx": {"signal": "(datum.leftside ? -1 : 1) * 6"},
              "angle": {"signal": "datum.leftside ? datum.angle - 180 : datum.angle"},
              "align": {"signal": "datum.leftside ? 'right' : 'left'"},
            }
          }
        }
      ]
    }

In [ ]:
def add_abks_to_graph(D, dataset='de', with_count=False, count_attr='binary'):
    abks = defaultdict(Counter)

    for node in reversed(list(nx.bfs_tree(D, 'root').nodes)):
        if D.out_degree(node) == 0:
            if dataset == 'de':
                 abk = node.split('_')[1]
            else:
                abk = node
            abks[node][abk] += 1 if count_attr == 'binary' else D.nodes[node][count_attr]


        parents = list(D.predecessors(node))
        if parents:
            abks[parents[0]].update(abks[node])
    
#     abks_counter = {
#         k: ', '.join(abk for abk, count in v.most_common())
#         for k, v in abks.items()
#     }
#     abks_counter = {
#         k: truncate(v, 100)
#         for k, v in abks_counter.items()
#     }
    abks_counter = {
        k: [
            f'{abk} ({count})' if with_count else abk
            for abk, count in v.most_common()
        ]
        for k, v in abks.items()
    }
    nx.set_node_attributes(D, abks_counter, 'abks')

In [ ]:
def remove_names_from_intermediate_nodes(data, D):
    not_leaves = {x['parent'] for x in data if x['id'] != 'root'}
    for row in data:
        if row['id'] == 'root':
            row['name'] = 'root'
        elif row['id'] in not_leaves:
            del row['name']
    

In [ ]:
def sortable_value(text):
    match = re.fullmatch('\d+([a-z]*)', text, flags=re.IGNORECASE)
    if not match:
        return text.zfill(4)
    extra_len = len(match[1])
    return text.zfill(4 + extra_len)

In [ ]:
def abbreviate_buch(elem):
    if elem.get("name"):
        elem = elem.copy()
        elem["name"] = elem["name"] \
            .replace("Buch", "B.") \
            .replace("Erstes ", "1. ") \
            .replace("Zweites ", "2. ") \
            .replace("Drittes ", "3. ") \
            .replace("Viertes ", "4. ") \
            .replace("Fünftes ", "5. ")
        elem["name"] = truncate(elem["name"], 20)
    return elem

# Vergleich der Modelle

## Kookkurrenzen (Louvain, Gliederung in typische Rechtsbereiche)

In [ ]:
clustering = get_clustering_result(
    "2019-01-01_0-0_1-0_-1_o-1-0_t-paragraph_a-louvain_m1-0_s0.json", 
    'de', 
    'clustering',
    path_prefix='../'
)
add_community_to_graph(clustering)

In [ ]:
latex = clustering_to_community_abk_latex(clustering)
with open('../tables/meso_dendrogram_louvain_de_2019_cooccurrence_markov_1_0.tex', 'w') as f:
    f.write(latex)
print(latex)

# Kombinationsmodell

In [ ]:
G_de_2019_louvain_mixed = nx.read_gpickle(
    "../../legal-networks-data/de/11_cluster_results/"
    "2019-01-01_0-0_1-0_-1_o-2-0_t-paragraph_a-louvain_m0-7_s0.gpickle.gz"
)
data = graph_to_vega_data(G_de_2019_louvain_mixed, 'de', min_size=9000, size_attr='tokens_n', truncate_len=999)
data = [abbreviate_buch(elem) for elem in data]
chart = alt.Vega(vega_circle_dendrogram_schema(data, 0, rotation_angle=-10, default_radius=271))
used_abks = {d['name'].split(',')[0] for d in data if d.get('name')}
save_chart_and_crop(chart, 'meso_dendrogram_louvain_de_2019_mixed', used_abks)

In [ ]:
chart = alt.Vega(vega_circle_dendrogram_schema(data, 0, greyscale=True, rotation_angle=-10, default_radius=271))
save_chart_and_crop(chart, 'meso_dendrogram_louvain_de_2019_mixed_graycolor', used_abks)

## Vergleich des Detailgrads

In [ ]:
clustering = get_clustering_result(
    "2019-01-01_5-0_1-0_0_o-2-0_t-paragraph_a-louvain_m1-0_s0.json", 
    'de', 
    'clustering',
    path_prefix='../'
)
add_community_to_graph(clustering)

In [ ]:
abk_counter = Counter([v.split('_')[0] for v in nx.get_node_attributes(clustering.graph, 'citekey').values()])

content = ''

for idx, com in enumerate(clustering.communities):
    
    cite_keys = [
        (
            clustering.graph.nodes[n]['citekey'].split('_'), 
            clustering.graph.nodes[n].get('law_name') or ''
        )
        for n in com
        if 'citekey' in clustering.graph.nodes[n]
    ]
    cite_keys_formatted = defaultdict(list)
    lawnames = {}
    for (gesetz, nummer), lawname in sorted(cite_keys):
        lawnames[gesetz] = lawname
        cite_keys_formatted[gesetz].append(nummer)
    
    if len(cite_keys_formatted) > 1:
        content += f'\n\nCLUSTER {idx}\n'
        for gesetz in sorted(cite_keys_formatted.keys(), key=lambda x: -len(cite_keys_formatted[x])):
            
            if len(cite_keys_formatted[gesetz]) == abk_counter[gesetz]:
                contents = [f'\n\t===== Alle {abk_counter[gesetz]} Elemente =====']
            else:
                contents = sorted(cite_keys_formatted[gesetz], key=lambda x: sortable_value(x))
            content += f'- {gesetz} : {" ".join(contents)} \n\t{lawnames[gesetz]}\n'

with open('../tables/meso_de_2019_louvain_mixed_detailed.txt', 'w') as f:
    f.write(content)

print(content)